# Revenue Leakage Project — Milestone 1 ✅

Data Cleaning & Quality Audit | Python + Pandas

In [2]:
# Import libraries:

import pandas as pd
import numpy as np
import os

In [5]:
# Load the six datasets:

orders = pd.read_csv("dataset_raw/ecommerce_orders.csv")
customers = pd.read_csv("dataset_raw/ecommerce_customers.csv")
products = pd.read_csv("dataset_raw/ecommerce_products.csv")
order_items = pd.read_csv("dataset_raw/ecommerce_order_items.csv")
shipments = pd.read_csv("dataset_raw/ecommerce_shipments.csv")
returns_refunds = pd.read_csv("dataset_raw/ecommerce_returns_refunds.csv")

In [3]:
# Load the six datasets:

orders = pd.read_csv("dataset_cleaned/orders_clean.csv")
customers = pd.read_csv("dataset_cleaned/customers_clean.csv")
products = pd.read_csv("dataset_cleaned/products_clean.csv")
order_items = pd.read_csv("dataset_cleaned/order_items_clean.csv")
shipments = pd.read_csv("dataset_cleaned/shipments_clean.csv")
returns_refunds = pd.read_csv("dataset_cleaned/returns_refunds_clean.csv")

In [6]:
# Inspect every table:

tables = {
    "orders": orders,
    "customers": customers,
    "products": products,
    "order_items": order_items,
    "shipments": shipments,
    "returns_refunds": returns_refunds
}

for name, df in tables.items():
    print(name, df.shape)

orders (10000, 19)
customers (10500, 13)
products (1600, 9)
order_items (17000, 9)
shipments (10000, 13)
returns_refunds (5493, 16)


In [39]:
# Inspect columns:

for name, df in tables.items():
    print("\n", name)
    print(df.info(max_cols=None))


 orders
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   order_id              10000 non-null  int64  
 1   customer_id           10000 non-null  int64  
 2   order_date            10000 non-null  object 
 3   order_status          10000 non-null  object 
 4   payment_method        10000 non-null  object 
 5   subtotal              10000 non-null  float64
 6   discount_amount       10000 non-null  float64
 7   shipping_fee          10000 non-null  float64
 8   tax_amount            10000 non-null  float64
 9   total_order_value     10000 non-null  float64
 10  delivery_city         10000 non-null  object 
 11  delivery_state        10000 non-null  object 
 12  coupon_code           6277 non-null   object 
 13  is_failed_order       10000 non-null  object 
 14  is_returned           10000 non-null  object 
 15  revenue_lea

In [57]:
# Check missing values:

for name, df in tables.items():
    print("\n", name)
    print(df.isnull().sum())


 orders
order_id                   0
customer_id                0
order_date                 0
order_status               0
payment_method             0
subtotal                   0
discount_amount            0
shipping_fee               0
tax_amount                 0
total_order_value          0
delivery_city              0
delivery_state             0
coupon_code             3723
is_failed_order            0
is_returned                0
revenue_leakage_flag       0
refund_amount              0
revenue_loss_amount        0
net_revenue                0
dtype: int64

 customers
customer_id                0
customer_name              0
gender                     0
date_of_birth              0
city                       0
state                      0
pincode                    0
registration_date          0
acquisition_channel        0
customer_segment           0
is_repeat_customer         0
churn_risk                 0
customer_lifetime_value    0
dtype: int64

 products
product_id    

In [41]:
# Check duplicates:

for name, df in tables.items():
    print("\n", name)
    print(df.duplicated().sum())


 orders
0

 customers
0

 products
0

 order_items
0

 shipments
0

 returns_refunds
0


In [ ]:
# check primary-key duplicates:

In [42]:
orders["order_id"].duplicated().sum()

0

In [43]:
customers["customer_id"].duplicated().sum()

0

In [44]:
products["product_id"].duplicated().sum()

0

In [45]:
order_items["order_item_id"].duplicated().sum()

0

In [46]:
shipments["shipment_id"].duplicated().sum()

0

In [47]:
 returns_refunds["return_refund_id"].duplicated().sum()

0

# Cleaning

## 1. Missing values handling:

In [58]:
orders["coupon_code"].value_counts(dropna=False)

coupon_code
NaN           3723
PAYTM50       1302
BIGBASH       1257
WELCOME100    1251
FESTIVE20     1237
SAVE10        1230
Name: count, dtype: int64

In [8]:
# Decision: NaN means no coupon was used.

orders["coupon_code"] = orders["coupon_code"].fillna("No Coupon")
print(orders["coupon_code"].isnull().sum())

0


In [59]:
shipments["failure_reason"].value_counts(dropna=False)

failure_reason
NaN                     6975
Courier Delay            997
Customer Refused         420
Damaged Package          417
Customer Unavailable     406
Incorrect Address        398
Logistics Capacity       387
Name: count, dtype: int64

In [9]:
# Decision: most shipments don't have a failure reason.

shipments["failure_reason"] = shipments["failure_reason"].fillna("No Failure")
print(shipments["failure_reason"].isnull().sum())

0


In [60]:
shipments["delivery_status"].value_counts()

delivery_status
Delivered     6137
RTO           1362
Failed        1058
In Transit     838
Delayed        605
Name: count, dtype: int64

In [61]:
pd.crosstab(
    shipments["delivery_status"],
    shipments["actual_delivery_date"].isna()
)

actual_delivery_date,False,True
delivery_status,,
Delayed,605,0
Delivered,6137,0
Failed,0,1058
In Transit,0,838
RTO,0,1362


A missing actual delivery date represents an order that hasn't successfully completed delivery.

Do NOT fill these with today's date, 0, or another artificial value.

Leave them as NaN/NaT.

In [62]:
returns_refunds["refund_status"].value_counts()

refund_status
Processed    4382
Pending       839
Failed        272
Name: count, dtype: int64

In [63]:
pd.crosstab(
    returns_refunds["refund_status"],
    returns_refunds["refund_processed_date"].isna()
)

refund_processed_date,False,True
refund_status,,
Failed,262,10
Pending,808,31
Processed,4227,155


refund_processed_date contains 196 missing values, including 155 records marked Processed. These were retained as nulls because the source data does not provide sufficient information to reconstruct the missing dates.

In [64]:
shipments["delay_days"].value_counts().sort_index()

delay_days
-1    1130
 0    1200
 1    1051
 2    1074
 3    1146
 4    1120
 5    1066
 6    1111
 7    1102
Name: count, dtype: int64

In [65]:
(shipments["delay_days"] == -1).sum()

1130

In [66]:
pd.crosstab(
    shipments["delivery_status"],
    shipments["delay_days"] == -1
)

delay_days,False,True
delivery_status,,
Delayed,529,76
Delivered,5476,661
Failed,950,108
In Transit,733,105
RTO,1182,180


In [ ]:
# Delay cleaning- preserve the original column and create a cleaned analytical column:

In [10]:
shipments["delay_days_clean"] = shipments["delay_days"].replace(-1, np.nan)
print(shipments["delay_days_clean"].isnull().sum())

1130


In [11]:
shipments["delay_days_valid"] = shipments["delay_days"] != -1
print(shipments["delay_days_valid"].value_counts())

delay_days_valid
True     8870
False    1130
Name: count, dtype: int64


In [12]:
tables["shipments"].columns

Index(['shipment_id', 'order_id', 'shipment_date', 'expected_delivery_date',
       'actual_delivery_date', 'courier_partner', 'warehouse_city',
       'delivery_status', 'delivery_attempts', 'shipping_cost',
       'failure_reason', 'delay_days', 'is_late_delivery', 'delay_days_clean',
       'delay_days_valid'],
      dtype='object')

## 2. Date columns datatype conversion:

In [13]:
# identifying Date columns

for name, df in tables.items():
    print("\n", name)
    print([col for col in df.columns if "date" in col.lower()])


 orders
['order_date']

 customers
['date_of_birth', 'registration_date']

 products
['launch_date']

 order_items
[]

 shipments
['shipment_date', 'expected_delivery_date', 'actual_delivery_date']

 returns_refunds
['return_request_date', 'refund_request_date', 'refund_processed_date']


In [14]:
# Convert date columns Data type:

for name, df in tables.items():

    date_cols = [
        col for col in df.columns
        if "date" in col.lower()
    ]

    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")

In [83]:
# Verify the date columns conversion:

for name, df in tables.items():

    print("\n", name)

    date_cols = [
        col for col in df.columns
        if "date" in col.lower()
    ]

    for col in date_cols:
        print(col, "→", df[col].dtype)


 orders
order_date → datetime64[ns]

 customers
date_of_birth → datetime64[ns]
registration_date → datetime64[ns]

 products
launch_date → datetime64[ns]

 order_items

 shipments
shipment_date → datetime64[ns]
expected_delivery_date → datetime64[ns]
actual_delivery_date → datetime64[ns]

 returns_refunds
return_request_date → datetime64[ns]
refund_request_date → datetime64[ns]
refund_processed_date → datetime64[ns]


# Validation

In [91]:
# Check row counts again:

for name, df in tables.items():
    print(name, df.shape)

orders (10000, 19)
customers (10500, 13)
products (1600, 9)
order_items (17000, 9)
shipments (10000, 15)
returns_refunds (5493, 16)


In [92]:
# Check missing values again:

for name, df in tables.items():
    print("\n", name)
    print(df.isnull().sum()[df.isnull().sum() > 0])


 orders
Series([], dtype: int64)

 customers
Series([], dtype: int64)

 products
Series([], dtype: int64)

 order_items
Series([], dtype: int64)

 shipments
actual_delivery_date    3258
delay_days_clean        1130
dtype: int64

 returns_refunds
refund_processed_date    196
dtype: int64


In [93]:
# Check for unique primary keys again:

primary_keys = {
    "orders": "order_id",
    "customers": "customer_id",
    "products": "product_id",
    "order_items": "order_item_id",
    "shipments": "shipment_id",
    "returns_refunds": "return_refund_id"
}

for name, key in primary_keys.items():
    print(
        name,
        "duplicates:",
        tables[name][key].duplicated().sum()
    )

orders duplicates: 0
customers duplicates: 0
products duplicates: 0
order_items duplicates: 0
shipments duplicates: 0
returns_refunds duplicates: 0


In [95]:
# Validate foreign keys for one of the many relationships: 
# child["foreign_key"].isin(parent["primary_key"]).value_counts()

orders["customer_id"].isin(customers["customer_id"]).value_counts()

customer_id
True    10000
Name: count, dtype: int64

In [96]:
# verify whether each order has exactly one shipment:
shipments["order_id"].duplicated().sum()

0

# Saving the cleaned dataset:

In [15]:
output_path = "dataset_cleaned"

os.makedirs(output_path, exist_ok=True)

for name, df in tables.items():
    df.to_csv(
        f"{output_path}/{name}_clean.csv",
        index=False
    )

In [98]:
os.listdir(output_path)

['customers_clean.csv',
 'orders_clean.csv',
 'order_items_clean.csv',
 'products_clean.csv',
 'returns_refunds_clean.csv',
 'shipments_clean.csv']

# Fixing shipments table 

There are three issues:

True/False → MySQL BOOLEAN import wizard is expecting 0/1.

Empty actual_delivery_date → MySQL DATE doesn't accept ''; it needs NULL.

Empty delay_days_clean → same issue; it needs NULL.

In [25]:
# Convert delay_days_valid to 0/1:

shipments["delay_days_valid"] = (
    shipments["delay_days_valid"].astype(int)
)

In [26]:
print(shipments["delay_days_valid"].unique())

[1 0]


In [27]:
# making sure the CSV uses empty cells for missing values:

shipments.to_csv(
    f"{output_path}/shipments_clean.csv",
    index=False,
    na_rep=""
)

# Then in MySQL, use LOAD DATA and explicitly tell MySQL to convert empty fields to NULL.

In [ ]:
# return_refund_error

In [4]:
print(len(returns_refunds))

print(
    returns_refunds["refund_processed_date"].isna().sum()
)

5493
196


In [6]:
import pandas as pd

returns_check = pd.read_csv(
    f"dataset_cleaned/returns_refunds_clean.csv"
)

print(returns_check.shape)

print(
    returns_check["refund_processed_date"].isna().sum()
)

(5493, 16)
196
